# 14 - Train Offline DQN with a Value-Gap Trace Cut

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`. The backup is still `DqnObjective`. The continuation is `gate=value_gap_gate(beta=BETA, normalize=True, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED, eps=EPS)`: the trace is cut by how far the taken action sits below `Q(s, a*)`, relative to the spread of the value Q at that state. `policy_delayed` picks `a* = argmax` from online or delayed Q. `value_delayed` scores that action and the taken action on online or delayed Q. Either flag, or both, reads the delayed net.

At the state the action is taken from, `V = max_a Q(s, a)` and

`c = exp(-beta * (V - Q(s, a_taken)) / (max_a Q(s, a) - min_a Q(s, a) + eps))`

`V - Q = 0` when the value Q scores the taken action at least as high as `a*`, so `c = 1` and the return keeps going, the same as a `λ = 1` step. When both flags agree, that is a tie with `max_a Q`. `normalize=True` divides by the range of the value Q, which makes `beta` dimensionless: a taken action at the minimum continues near `exp(-beta)` when `a*` is a max and that range is large next to `eps`. `normalize=False` turns that term off and uses the raw gap, `c = exp(-beta * (V - Q(s, a_taken)))`, and takes no `eps`. A larger gap shrinks `c` toward `0`, and the return bootstraps the delayed state value at that state. The coefficient is a constant in the TD error. `watkins_gate()` is the hard version of the online cut (`c` is `0` or `1`).

The gate returns `[N, N]`. Row `t`, column `s` is the continuation at step `s` for the return that started at `t` — the action taken from `s`, which the batch stores on the next row. Entries with `s <= t` are unused. A run break (`sequence_id` / `grouping_field`) or a `0` discount still ends the trace.

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from a pretrained backbone (token embeddings included) and an action-value head.
4. Train with `DqnObjective(gate=value_gap_gate(beta=BETA, normalize=True, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED, eps=EPS))` and save with `push_model_to_hub`.

Each step is packed by `Tokenizer` as text and embedded by the backbone's `embed_tokens`.

- `type: "text"` — each field has its own `format=` and is tokenized as its own run. With `input_field=`, `format=` has exactly one placeholder `{field}` (a spec such as `{field:g}` is allowed). Omit `input_field=` for a const: keep `output_field=` and set `format=` to the literal string (no placeholder).
- `type: "token"` — integer id → one `embed_tokens` row (raw vocab id; not used here)
- `type: "image"` — vision span (VL checkpoints; not used here)

There is no whole-step `format=`. Fields emit in `input_fields` order, each as a separate HF tokenize call, so BPE never merges across fields. `group_prefix=` is tokenized as `__text__` and inserted by `pack_token_batch` at the start of every `task_index` segment (and at the start of each packed sequence). Incremental decode that packs only new steps must pass `prev_grouping_ids` (a required argument of `Tokenizer.pack_rows`) so a cached task does not re-emit the group prefix. Reward `0.0` and done codes `0` use `skip` / `format_skipped=""` so a zero value emits nothing. `task_done` is an objective column only. `value` is a text const (`output_field="value"`, `format="\n"`, `max_tokens=1`) flagged `head_output: True`: the newline that ends each step row is the Q readout position; the tokenizer raises if that field emits more than one token.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount, value_gap_gate
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-value-gap-offline"      # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-value-gap-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
BETA = 1.0                                    # scale on the gap
NORMALIZE = True                              # True divides by max_a Q - min_a Q + EPS on the value Q
POLICY_DELAYED = False                        # True picks a* from delayed Q
VALUE_DELAYED = False                         # True scores V and the taken action on delayed Q
EPS = 1e-6                                    # floor on that range; required when NORMALIZE is True
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "skip": 0.0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "skip": 0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
    group_prefix="action,observation,r=reward,d=done\n",
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves. `use_norm` (required, saved with the model) keeps (`True`) or drops (`False`) the final RMSNorm.
- `RegressionHead` predicts one value per discrete action. `use_norm` (required) keeps (`True`) or drops (`False`) the head's input RMSNorm.

The backbone exposes `hidden_dim`, and the head uses that same value so the pieces connect cleanly.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. Pass `heads=` as a dict that names each head (`{"action_value": head}` here). `out.predictions["action_value"]` is that head's tensor; `action_source="action_value"` is the name `get_action` reads.

### What packed steps look like

Three steps (each text field is its own tokenize run; reward / episode_done skipped when zero):

```text
step 0:  action=0, observation=1, reward=0.0, episode_done=0
step 1:  action=2, observation=5, reward=0.0, episode_done=0
step 2:  action=1, observation=7, reward=1.0, episode_done=1
```

Each field fills its own `format=` (`{field}` for action, `,{field}` for observation, `,r={field:g}` for reward, `,d={field}` for episode_done, and the const `\n` for `value`). `{field:g}` drops a trailing `.0`. Reward / episode_done use `format_skipped=""` so a zero value emits nothing.
`group_prefix=` names the columns once at the start of the `task_index` segment (here all three steps share task 0).
`*\n*` marks the head-output token (the row-ending newline) — the head reads Q from it to score the next action.

```text
task=0
action,observation,r=reward,d=done
0,1,*\n*
2,5,*\n*
1,7,r=1,d=1,*\n*
```

Packed in order: group prefix · step0 · step1 · step2. `head_output_indices` points at every `*\n*` token.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)


head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
)

model = Model(
    backbone=backbone,
    heads={"action_value": head},
    action_source="action_value",
    reasoner=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data=objective_data, predictions=q, delayed_predictions=q_target)` computes the DQN loss and metrics. Pass `out.predictions["action_value"]` — the name given in `heads=`.
4. `AdamW` updates weights. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q comes from the delayed model: `delayed_model = model.copy(heads=True, backbone=True, reasoner=False)` copies every head and the fp32 backbone (including token embeddings). After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each copied section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`value_gap_gate(beta=BETA, normalize=NORMALIZE, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED, eps=EPS)` is the value-gap continuation when `NORMALIZE` is `True`. `normalize=False` is `value_gap_gate(beta=BETA, normalize=False, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED)`, with no `eps`. `policy_delayed` picks `a*`. `value_delayed` scores `V = Q(s, a*)` and the taken action. Either flag, or both, reads detached delayed Q. Along a run

`G_t = r_{t+1} + γ_{t+1} ((1 - c_{t+1}) V(s_{t+1}) + c_{t+1} G_{t+1})`

with `c_s = exp(-BETA * (V - Q(s, a_s)) / (max_a Q - min_a Q + EPS))` when `NORMALIZE` is `True`, on the value Q after `value`. `V = Q(s, a*)` and `a*` is the argmax of the policy Q. `NORMALIZE = False` drops the denominator and uses `c_s = exp(-BETA * (V - Q(s, a_s)))`, and that call omits `eps`. The `max` and `min` are the action values of the value Q at `s`. A negative raw gap is floored at 0, so `c = 1`. `V` in the backup is the delayed state value of this head: `max Q` when `temperature=0`, or the SAC soft value `α logsumexp(Q / α)` when `temperature=α > 0`. A zero gap at `s_{t+1}` carries `G_{t+1}` in full. A large gap drops that term and bootstraps `V(s_{t+1})`.

`DqnObjective` takes `discount`, called with the unpacked `objective_data` columns (`episode_done` / `task_done` are `0`/`1`/`2`). `boundary_discount` is the standard lookup: `gamma_step` always multiplies, then episode and task extras (`1.0` when the matching code is `0`). When a task ends both extras fire, so a task extra of `0.0` zeros the whole term. `reward=None` and `value=None` skip those callables (raw `reward` column, raw Q). Pass `boundary_reward` / `boundary_value` to transform them. Required `temperature` is the SAC / soft-Q `α`: `0` is hard max-Q; `> 0` bootstraps from `α logsumexp(Q / α)` (same meaning as `get_action(temperature=)`). Required `double`: `False` reads that bootstrap from delayed Q; `True` is Double DQN, where detached online Q chooses the action and delayed Q scores it. The gamma at each step multiplies both the bootstrap and the continued return, so a `0` gamma ends the trace and a non-zero truncation gamma carries it, discounted. The trace never reads across a run break (`sequence_id` / `grouping_field`).


In [ ]:
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.copy(heads=True, backbone=True, reasoner=False)
polyak = Polyak(online=model, delayed=delayed_model)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_episode_terminal=1.0,
        gamma_episode_truncated=1.0,
        gamma_task_terminal=0.0,
        gamma_task_truncated=0.0,
    ),
    grouping_field="task_index",
    temperature=0.0,
    double=False,
    gate=(
        value_gap_gate(beta=BETA, normalize=True, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED, eps=EPS)
        if NORMALIZE
        else value_gap_gate(beta=BETA, normalize=False, policy_delayed=POLICY_DELAYED, value_delayed=VALUE_DELAYED)
    ),
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        loss, metrics = objective(
            objective_data=to_device(
                data=objective_data, device=device),
                predictions=out.predictions["action_value"],
                delayed_predictions=delayed_out.predictions["action_value"],
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")